# 第2周 · Python II：函数、文件、NumPy 与绘图

**本周结束时你能做到**：统计莎士比亚全部台词的词频；用 NumPy 一行代码完成上周用循环写的成绩统计；把手写数字图片当作数组处理；画出规范的科研图。

本周会第一次见到“张量”：一个形状为 `(1797, 8, 8)` 的数组——1797 张 8×8 像素的手写数字图片。之后所有深度学习模型吃进去的，都是这种东西。

In [ ]:
# 运行这一格来安装课程工具包（每次在 Colab 打开 notebook 后运行一次）
import importlib.util, subprocess, sys
if importlib.util.find_spec("zero2researcher") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/ReaganGen/Zero2Researcher"], check=True)
print("课程工具包已就绪")

In [ ]:
from zero2researcher import data
from zero2researcher.checks import week02 as ck

---
## S1 · 函数

函数是一段有名字、可以反复调用的代码，就像一份标准操作流程（SOP）：输入样品（参数），按步骤处理，交出结果（`return`）。

```python
def 函数名(参数1, 参数2):
    """一句话说明这个函数做什么"""
    ...
    return 结果
```

In [ ]:
def celsius_to_fahrenheit(c):
    """把摄氏度换算成华氏度"""
    return c * 9 / 5 + 32

print(celsius_to_fahrenheit(37))
ck.celsius_to_fahrenheit(celsius_to_fahrenheit)    # 注意：传入函数本身，不加括号

### 写 `is_palindrome(s)`

判断一句话是不是回文：先转小写、去掉所有非字母字符，再与反转后的自己比较。`ch.isalpha()` 判断一个字符是不是字母。

In [ ]:
def is_palindrome(s):
    """忽略大小写和非字母字符，判断是否为回文"""
    letters = ""
    for ch in s.lower():
        if ch.isalpha():
            letters += ch
    return letters == letters[::-1]

print(is_palindrome("A man, a plan, a canal: Panama"))
ck.is_palindrome(is_palindrome)

### 写 `clean_word(word)`

统计词频前要先“清洗”每个词：转小写，只保留字母。例如 `"ROMEO:"` → `"romeo"`，`"don't"` → `"dont"`。

In [ ]:
def clean_word(word):
    """转小写并只保留字母"""
    out = ""
    for ch in word.lower():
        if ch.isalpha():
            out += ch
    return out

print(clean_word("Hello,"))
ck.clean_word(clean_word)

### 默认参数

给参数一个默认值，调用时可以省略它。下面把上周的等级判断写成函数，及格线可以调整：

In [ ]:
def letter_grade(score, pass_mark=60):
    if score >= 90:
        return "A"
    elif score >= 80:
        return "B"
    elif score >= 70:
        return "C"
    elif score >= pass_mark:
        return "D"
    return "F"

print(letter_grade(65))
print(letter_grade(65, pass_mark=70))

### 作用域：函数内部的变量在外面看不见

函数里创建的变量是“局部”的，函数结束就消失。需要的结果一定要 `return` 出来。

In [ ]:
def average(values):
    total = sum(values)
    return total / len(values)

print(average([80, 90, 70]))
print(total)

---
## S2 · 读写文件：莎士比亚的词频

`tinyshakespeare.txt` 收录了莎士比亚多部戏剧的台词，约 4 万行、111 万个字符（公有领域文本）。第12周你会用同一份文本训练一个会“写莎士比亚”的语言模型。

In [ ]:
text_path = data.path("tinyshakespeare.txt")
with open(text_path) as f:
    for i, line in enumerate(f):
        print(repr(line))
        if i == 5:
            break

注意每行末尾的 `\n`（换行符）。三个最常用的字符串方法：
- `line.strip()`：去掉首尾空白和换行符
- `line.startswith("ROMEO")`：是否以某段文字开头
- `line.split()`：按空白切开成列表

In [ ]:
line = "First Citizen:\n"
print(line.strip())
print(line.split())
print([clean_word(w) for w in "Speak, speak.".split()])

### 统计词频

逐行读入，每行用 `.split()` 切成词，用 `clean_word` 清洗，清洗后为空的跳过，用字典计数。

In [ ]:
counts = {}
with open(text_path) as f:
    for line in f:
        for token in line.split():
            w = clean_word(token)
            if w:
                counts[w] = counts.get(w, 0) + 1
print(len(counts), "个不同的词")
print(counts["romeo"], counts["juliet"], counts["king"])

### 按次数排序，写出前50个高频词

In [ ]:
import os
os.makedirs("outputs", exist_ok=True)

ranked = sorted(counts.items(), key=lambda kv: (-kv[1], kv[0]))   # 次数从大到小；次数相同按字母
print(ranked[:10])

with open("outputs/top_words.csv", "w") as f:
    f.write("word,count\n")
    for w, n in ranked[:50]:
        f.write(f"{w},{n}\n")
ck.top_words_csv("outputs/top_words.csv")

---
## S3 · NumPy：从列表到数组

上周我们用了很多 `for` 循环。NumPy 数组让我们**一次对整批数字做运算**，既简洁又快。

In [ ]:
import numpy as np

scores = np.array(data.load_scores(), dtype=float)
print(scores.shape, scores.dtype, scores.ndim)
ck.scores_array(scores)

### 索引与切片：`数组[行, 列]`

In [ ]:
print(scores[2, 2])        # 第3位同学的物理
print(scores[:5, 0])       # 前5位同学的数学
print(scores[:, 1].shape)  # 全部同学的英语

### 布尔掩码：一行代码找出录入错误

In [ ]:
bad = (scores < 0) | (scores > 100)
print(bad.sum(), "个录入错误")
print(np.argwhere(bad))     # 它们在哪一行、哪一列
ck.invalid_mask(bad)

把错误成绩换成 `np.nan`（“不是一个数”），之后用 `np.nanmean` 计算时会自动跳过它们。

**`axis` 参数**：`axis=0` 沿着行的方向压缩（对每一列、也就是每门课求平均），`axis=1` 沿着列的方向压缩（对每一行、也就是每位同学求平均）。

In [ ]:
clean = scores.copy()
clean[bad] = np.nan
subject_means = np.nanmean(clean, axis=0)
print(np.round(subject_means, 2))     # 上周十几行的循环，现在一行
ck.subject_means(subject_means)

### 广播：形状兼容的数组直接运算

按科目做 z-score 标准化：每一列减去自己的均值、除以自己的标准差。`clean` 是 (30, 3)，`np.nanmean(clean, axis=0)` 是 (3,)，NumPy 会自动把后者“广播”到每一行。

In [ ]:
z = (clean - np.nanmean(clean, axis=0)) / np.nanstd(clean, axis=0)
print(np.round(z[:5], 2))
ck.zscores(z)

### 向量化有多快？

In [ ]:
x = np.random.rand(100_000)

def sum_squares_loop(values):
    total = 0.0
    for v in values:
        total += v * v
    return total

%timeit sum_squares_loop(x)
%timeit (x ** 2).sum()

### 图片就是数组：你的第一个张量

scikit-learn 自带一个手写数字数据集：1797 张 8×8 像素的灰度图片，每个像素是 0–16 的整数。

In [ ]:
from sklearn.datasets import load_digits

digits = load_digits()
images = digits.images
labels = digits.target
print(images.shape)     # (图片数, 高, 宽)
print(labels[:10])
print(images[0])        # 第一张图片，就是一个 8x8 的数字矩阵
ck.digits_shape(images)

`(1797, 8, 8)`——记住这种形状。第8周用 MNIST（28×28）训练神经网络、第10周用 CIFAR-10（32×32、3个颜色通道）训练CNN，读进去的都是这种张量。

---
## S4 · Matplotlib 作图

一张合格的科研图：**轴标签带单位、有图例、误差或分布可见、标题说出结论**。

In [ ]:
import matplotlib.pyplot as plt

### 折线图：一年的日均气温

In [ ]:
temps = data.load_temperature()
fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(range(1, 366), temps, linewidth=1)
ax.set_xlabel("Day of year")
ax.set_ylabel("Daily mean temperature (°C)")
ax.set_title("Temperatures peak in July and bottom out in January")
plt.show()

### 直方图：数学成绩分布（只用有效成绩）

In [ ]:
fig, ax = plt.subplots(figsize=(5, 3))
math_valid = clean[:, 0][~np.isnan(clean[:, 0])]
ax.hist(math_valid, bins=10, edgecolor="white")
ax.axvline(math_valid.mean(), color="k", linestyle="--", label=f"mean = {math_valid.mean():.1f}")
ax.set_xlabel("Math score")
ax.set_ylabel("Number of students")
ax.legend()
plt.show()

### 柱状图：莎士比亚最常用的20个词

In [ ]:
top20 = ranked[:20]
fig, ax = plt.subplots(figsize=(8, 3))
ax.bar([w for w, n in top20], [n for w, n in top20])
ax.set_ylabel("Count")
ax.set_title("Function words dominate Shakespeare's vocabulary")
ax.tick_params(axis="x", rotation=60)
plt.show()

### 图片：每个数字各显示一张

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(7, 3))
for k, ax in enumerate(axes.flat):
    first = np.where(labels == k)[0][0]       # 第一个标签为 k 的位置
    ax.imshow(images[first], cmap="gray_r")
    ax.set_title(str(k))
    ax.axis("off")
plt.show()

### 本周小结

- 函数：`def`、参数、默认参数、`return`、局部变量
- 文件：`with open(...)`、`strip`、`split`；用字典统计词频
- NumPy：`shape`、索引与切片、布尔掩码、`np.nan`、`axis`、广播
- 图片就是数组：`(图片数, 高, 宽)` 的三维张量
- 作图规范：单位、图例、分布可见、有结论的标题